## Bronze ingestion below - > converted csv to delta table 


In [0]:
from pyspark.sql import functions as F

df_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("/Volumes/personal_projects/information_retail_schema/retailtransactions"))

# Clean column names by replacing spaces with underscores
for col in df_raw.columns:
    df_raw = df_raw.withColumnRenamed(col, col.replace(" ", "_"))

df_raw.write.format("delta").mode("overwrite").saveAsTable("personal_projects.information_retail_schema.online_retail_raw")

EDA

In [0]:
from pyspark.sql import functions as F

df = spark.table("personal_projects.information_retail_schema.online_retail_raw")

In [0]:
display(df.limit(10))

In [0]:
df

In [0]:
df.select("Country").distinct().count()

In [0]:
display(df.select("Country").distinct())

## Silver transformation of table 
- creates new table after tranformation of data 

In [0]:
from pyspark.sql import functions as F

df = spark.table("personal_projects.information_retail_schema.online_retail_raw")

df_clean = (df
    .filter(F.col("Quantity") > 0)               # drop returns/cancellations
    .filter(F.col("Price") > 0)
    .filter(F.col("Customer_ID").isNotNull())      # drop anonymous transactions
    .dropDuplicates(["Invoice", "StockCode", "Customer_ID"])
    .withColumn("InvoiceDate", F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm"))
    .withColumn("Revenue", F.col("Quantity") * F.col("Price"))
    .withColumnRenamed("Customer_ID", "CustomerID")
)

df_clean.write.format("delta").mode("overwrite").saveAsTable("personal_projects.information_retail_schema.silver_online_retail_clean")

In [0]:
from pyspark.sql import functions as F

df = spark.table("personal_projects.information_retail_schema.silver_online_retail_clean")

# Monthly revenue by country
monthly_revenue = (df
    .withColumn("YearMonth", F.date_format("InvoiceDate", "yyyy-MM"))
    .groupBy("YearMonth", "Country")
    .agg(F.round(F.sum("Revenue"), 2).alias("TotalRevenue"),
         F.countDistinct("Invoice").alias("NumOrders")))

monthly_revenue.write.format("delta").mode("overwrite").saveAsTable("personal_projects.information_retail_schema.gold_monthly_revenue_by_country")

# Customer lifetime value
customer_ltv = (df.groupBy("CustomerID")
    .agg(F.round(F.sum("Revenue"),2).alias("TotalSpend"),
         F.countDistinct("Invoice").alias("NumOrders"),
         F.max("InvoiceDate").alias("LastPurchase"))
    .orderBy(F.desc("TotalSpend")))

customer_ltv.write.format("delta").mode("overwrite").saveAsTable("personal_projects.information_retail_schema.gold_customer_ltv")

# Top products
top_products = (df.groupBy("StockCode", "Description")
    .agg(F.round(F.sum("Revenue"),2).alias("TotalRevenue"),
         F.sum("Quantity").alias("TotalQuantity"))
    .orderBy(F.desc("TotalRevenue")))

top_products.write.format("delta").mode("overwrite").saveAsTable("personal_projects.information_retail_schema.gold_top_products")

In [0]:
revenue = spark.table("personal_projects.information_retail_schema.gold_monthly_revenue_by_country")
display(revenue)
display(df)

In [0]:
customer = spark.table("personal_projects.information_retail_schema.gold_customer_ltv")
display(customer)

In [0]:
top_products = spark.table("personal_projects.information_retail_schema.gold_top_products")
display(top_products)